# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to load, explore, and analyze the FAIR² dataset: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution.

### Dataset Source
The dataset uses the Croissant schema and is provided via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n\n{metadata['description']}\n\nPublished: {metadata.get('datePublished', 'Unknown')}\nIdentifier: {metadata.get('identifier', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in Croissant schema are referenced by `@id` field. Below, we list all record sets and their corresponding fields using their `@id` values.

In [ ]:
# Access dataset metadata for recordSet definitions
record_sets = []
fields_by_recordset = {}

# The metadata may have 'recordSet' key - list of record sets
meta = dataset.metadata.to_json()
record_sets_meta = meta.get('recordSet', [])
if not record_sets_meta:
    print('No record sets defined in the metadata.')
else:
    for rs in record_sets_meta:
        rs_id = rs.get('@id', None)
        if rs_id:
            record_sets.append(rs_id)
            # Each recordSet may have fields
            fields = rs.get('field', [])
            field_ids = [f.get('@id', None) for f in fields if isinstance(f, dict) and '@id' in f]
            fields_by_recordset[rs_id] = field_ids
    print('Record Sets and their Fields (@id):')
    for rs_id in record_sets:
        print(f"  Record Set: {rs_id}")
        print(f"    Fields: {fields_by_recordset.get(rs_id, [])}")

# If no recordSet definitions in metadata, try to infer from mlcroissant API
if not record_sets:
    # Try listing by dataset.records() with no arguments
    print('\nInferring available record sets from mlcroissant...')
    available = dataset.available_record_sets()
    record_sets = available
    for rs_id in record_sets:
        try:
            sample = next(dataset.records(record_set=rs_id))
            print(f"Record Set: {rs_id}")
            print(f"Sample fields: {list(sample.keys())}")
        except StopIteration:
            print(f"Record Set: {rs_id} (empty)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*All extraction is referenced using `@id` values.*

In [ ]:
# Extract data from each record set
dataframes = {}
# Choose the main record set - from Overview, or inferred
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Extracting records from main record set: {main_record_set_id}")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns in {main_record_set_id}:")
    print(df.columns.tolist())
    df.head()
else:
    print('No record sets to extract.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

*All field references use their `@id`.*

In [ ]:
# Choose a numeric field by @id
# For demonstration, search for a field likely representing age or interval (e.g., '@id': 'age', 'interval_between_cancers')
df = dataframes.get(main_record_set_id, pd.DataFrame())
if not df.empty:
    # Try to find a numeric field
    numeric_field_id = None
    for col in df.columns:
        # Try heuristics
        if 'age' in col.lower() or 'interval' in col.lower():
            numeric_field_id = col
            break
    # If not found, just pick the first numeric column
    if not numeric_field_id:
        for col in df.select_dtypes('number').columns:
            numeric_field_id = col
            break
    
    if numeric_field_id:
        # Example threshold
        threshold = 50
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:\n")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try group by another categorical field
        group_field_id = None
        for col in df.columns:
            if ('sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:\n")
            print(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of a numeric field or show relationships between MSI status and anatomical location, using `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id available, plot by group
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No data or numeric field available for visualization.')

## 6. Conclusion
In this notebook, we've demonstrated loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields as per the Croissant schema. This workflow enables consistent, reproducible analysis and supports FAIR data best practices.

- Dataset metadata and structure can be queried directly from the schema.
- Record sets and fields are referenced by their unique `@id`s.
- Data extraction, processing, and visualization are possible in a few simple steps.

Further investigations may include advanced statistical analysis or machine learning tasks using the extracted dataframes.